<a href="https://colab.research.google.com/github/Sulu421/BIFX-546_Project-Parkinson-s_Telemonitoring/blob/main/Notebooks/BIFX546_Project__%22Parkinson's_Telemonitoring%22_SulemanMohammad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **Project Title: Voice-Based Prediction of Parkinson’s Disease Severity**

##### **Student:       Suleman Mohammad**
##### **School:        Hood Graduate College, MS. Bioinformatics**
##### **Course:        BIFX-546 Machine Learning for Bioinformatics**
##### **Instructor:    Sarangan Ravichandran, Ph.D., PMP**    
##### **Dataset:       Parkinson's Telemonitoring**                                                               

**Disclaimer**: This repository contains analysis of my Final Project for my course  BIFX 546 – Machine Learning for Bioinformatics, focusing on the Parkinsons Telemonitoring dataset. The main goal of this project is to use biomedical voice features to predict clinical motor_UPDRS and total_UPDRS scores for patients with early-stage Parkinson’s disease.

#### =============================================================================
#### 1. DATASET LOADING
#### =============================================================================

##### LOADING THE REQUIRED PACKAGES

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

##### ADDITIONAL PACKAGES WHICH ARE REQUIRED TO LOAD DATASET DIRECTLY FROM THE UCI REPO:

In [ ]:
#| echo: false
##To install the ucimlrepo package, you can use the following command in your terminal or command prompt:
%pip install ucimlrepo

##In Jupyter Notebook, you can use the following command to install the ucimlrepo package:
#!pip3 install -U ucimlrepo

from ucimlrepo import fetch_ucirepo

#####  LOADING THE DATASET USING DIFFERENT APPROACHES

**NOTE:** Select and implement the approach that best aligns with your methodology. Comment out any alternative approaches not used in the final implementation.

In [ ]:
# 1. Load the dataset from the local CSV file
#df = pd.read_csv('Pakisons-Data-File.csv')

# 2. Load the dataset from the local excel file
#df = pd.read_excel('Pakisons-Data-File.xlsx')

# 3. Load dataset using ucimlrepo package (recommended approach)
parkinsons_telemonitoring = fetch_ucirepo(id=189)

# metadata
print(parkinsons_telemonitoring.metadata)
# variable information
print(parkinsons_telemonitoring.variables)
# Create a working copy of the dataset
df = parkinsons_telemonitoring.data["original"].copy()

# Basic structure
df_shape = df.shape
n_patients = df["subject#"].nunique()
n_records = len(df)
print(f"Dataset shape: {df_shape}")
print(f"Unique patients: {n_patients} | Total recordings: {n_records}")


#### =============================================================================
#### 2. INITIAL DATA CLEANING
#### =============================================================================

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()

# Check for duplicates
duplicates = df.duplicated().sum()

# Check data types
print(df.dtypes)

# Drop the unnecessary index column
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])
df.columns

#### =============================================================================
#### 3. SUMMARY STATISTICS
#### =============================================================================

In [ ]:
# Basic descriptive statistics
summary = df.describe()

# ── Target Variable statistics ────────────────────────────────────────────
motor_stats = {
    "mean":   df["motor_UPDRS"].mean(),
    "median": df["motor_UPDRS"].median(),
    "std":    df["motor_UPDRS"].std(),
    "min":    df["motor_UPDRS"].min(),
    "max":    df["motor_UPDRS"].max()
}

total_stats = {
    "mean":   df["total_UPDRS"].mean(),
    "median": df["total_UPDRS"].median(),
    "std":    df["total_UPDRS"].std(),
    "min":    df["total_UPDRS"].min(),
    "max":    df["total_UPDRS"].max()
}

# ── Demographics ──────────────────────────────────────────────────────────
Age_mean = df["age"].mean()
Age_min  = df["age"].min()
Age_max  = df["age"].max()

# FIX: use plain assignment, NOT set literal {} — the original code stored
# a Python set {number} instead of the integer itself.
Male   = (df['sex'] == 0).sum()
Female = (df['sex'] == 1).sum()

# ── Recording Period ──────────────────────────────────────────────────────
duration_start = df['test_time'].min()
duration_end   = df['test_time'].max()
average_recordings_per_patient = len(df) / df['subject#'].nunique()

# ── Display all summary statistics ───────────────────────────────────────
print("="*60)
print("MOTOR UPDRS SUMMARY")
print("="*60)
for k, v in motor_stats.items():
    print(f"  {k:<8}: {v:.3f}")

print()
print("="*60)
print("TOTAL UPDRS SUMMARY")
print("="*60)
for k, v in total_stats.items():
    print(f"  {k:<8}: {v:.3f}")

print()
print("="*60)
print("DEMOGRAPHICS")
print("="*60)
print(f"  Age   : mean={Age_mean:.1f}  min={Age_min}  max={Age_max}")
print(f"  Male  : {Male} recordings  ({Male/len(df)*100:.1f}%)")
print(f"  Female: {Female} recordings  ({Female/len(df)*100:.1f}%)")

print()
print("="*60)
print("RECORDING PERIOD")
print("="*60)
print(f"  Recording period          : {duration_start:.1f} to {duration_end:.1f} days since recruitment")
print(f"  Avg recordings per patient: {average_recordings_per_patient:.1f}")


#### =============================================================================
#### 4. VISUALIZATIONS
#### =============================================================================

#### NOTE ON TARGET SELECTION AND HEATMAP SCOPE

**Target variables**: This project uses **two targets** — `motor_UPDRS` and `total_UPDRS`.
Because `total_UPDRS` includes `motor_UPDRS` as a major component, the two are highly correlated
(r ≈ 0.95). We treat **`motor_UPDRS` as the primary target** (it more directly reflects the acoustic
biomarkers of dysarthria) and `total_UPDRS` as a secondary target. Models are evaluated separately
for both to assess whether voice features generalise to the broader clinical score.

**Correlation heatmap**: The heatmap below includes **all 16 voice features** plus both targets.
The Jitter and Shimmer sub-families are highly inter-correlated (r > 0.80–0.91), which motivates
the use of Ridge/Lasso regularisation and Random Forest in the modelling phase.


In [ ]:
# GENERATING VISUALIZATIONS

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 12)

fig = plt.figure(figsize=(16, 12))

# ── Viz 1: Distribution of Motor UPDRS Scores ────────────────────────────
ax1 = plt.subplot(2, 2, 1)
sns.histplot(df['motor_UPDRS'], bins=30, kde=True, color='steelblue', ax=ax1)
ax1.set_title('Distribution of Motor UPDRS Scores', fontsize=14, fontweight='bold')
ax1.set_xlabel('Motor UPDRS Score', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.axvline(df['motor_UPDRS'].mean(), color='red', linestyle='--',
            linewidth=2, label=f'Mean: {df["motor_UPDRS"].mean():.2f}')
ax1.legend()

# ── Viz 2: Correlation Heatmap — ALL 16 voice features + both targets ────
# FIX: original code used only 7 of 16 features (Jitter/Shimmer sub-families
# were omitted). All 16 are included here so the multicollinearity structure
# is fully visible.
ax2 = plt.subplot(2, 2, 2)
all_voice_features = [
    'Jitter(%)', 'Jitter(Abs)', 'Jitter:RAP', 'Jitter:PPQ5', 'Jitter:DDP',
    'Shimmer', 'Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5',
    'Shimmer:APQ11', 'Shimmer:DDA',
    'NHR', 'HNR', 'RPDE', 'DFA', 'PPE',
    'motor_UPDRS', 'total_UPDRS'
]
correlation_matrix = df[all_voice_features].corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.6}, ax=ax2,
            annot_kws={"size": 6})
ax2.set_title('Correlation: All 16 Voice Features vs UPDRS', fontsize=13, fontweight='bold')
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right', fontsize=7)
plt.setp(ax2.get_yticklabels(), fontsize=7)

# ── Viz 3: Disease Progression Over Time ─────────────────────────────────
ax3 = plt.subplot(2, 2, 3)
sample_patients = df['subject#'].unique()[:5]
for patient in sample_patients:
    patient_data = df[df['subject#'] == patient].sort_values('test_time')
    ax3.plot(patient_data['test_time'], patient_data['motor_UPDRS'],
             marker='o', markersize=3, alpha=0.7, label=f'Patient {patient}')
ax3.set_title('Motor UPDRS Progression Over Time (Sample)', fontsize=14, fontweight='bold')
ax3.set_xlabel('Days Since Recruitment', fontsize=12)
ax3.set_ylabel('Motor UPDRS Score', fontsize=12)
ax3.legend(loc='best', fontsize=9)
ax3.grid(True, alpha=0.3)

# ── Viz 4: Gender Comparison ──────────────────────────────────────────────
ax4 = plt.subplot(2, 2, 4)
df['Gender'] = df['sex'].map({0: 'Male', 1: 'Female'})
sns.boxplot(x='Gender', y='motor_UPDRS', data=df, ax=ax4)
ax4.set_title('Motor UPDRS Distribution by Gender', fontsize=14, fontweight='bold')
ax4.set_ylabel('Motor UPDRS Score', fontsize=12)
ax4.set_xlabel('Gender', fontsize=12)

plt.tight_layout()

# ── FIX: Save figures to Google Drive (persistent) with local fallback ────
import os
try:
    from google.colab import files
except ImportError:
    files = None

DRIVE_PATH = '/content/drive/MyDrive/BIFX546_Project'
FIG_NAME   = 'parkinsons_eda_visualizations.png'

if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(DRIVE_PATH, exist_ok=True)
    save_path = os.path.join(DRIVE_PATH, FIG_NAME)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Figure saved to Google Drive: {save_path}")
else:
    # Colab session storage — download immediately so it isn't lost
    plt.savefig(FIG_NAME, dpi=300, bbox_inches='tight')
    print("Google Drive not mounted. Downloading figure directly...")
    if files is not None:
        files.download(FIG_NAME)

plt.show()


#### =============================================================================
#### 5. MODELING — PATIENT-LEVEL TRAIN/TEST SPLIT (GroupShuffleSplit)
#### =============================================================================

> **⚠️ Critical methodological note**: This dataset contains ~140 recordings per patient.
> A **random** train/test split would place recordings from the same patient in both sets,
> causing the model to effectively memorise individual voice characteristics rather than
> learning to generalise to unseen patients — a form of **data leakage**.
>
> We use `GroupShuffleSplit` from scikit-learn, which guarantees that all recordings from
> a given patient are either entirely in train or entirely in test. This produces a
> realistic estimate of how the model would perform on a **new, unseen patient**.
>
> **Primary target**: `motor_UPDRS` (most directly tied to vocal biomarkers of dysarthria).
> **Secondary target**: `total_UPDRS` (evaluated separately to test generalisation).


In [ ]:
# install scikit-learn if not already installed using following command in terminal or command prompt

# !pip3 install scikit-learn

#  Then verify it worked in a new cell:

import sklearn
print(sklearn.__version__)

# Then import these packages:

from sklearn import model_selection
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# ── Feature / target setup ────────────────────────────────────────────────
FEATURE_COLS = [
    'Jitter(%)', 'Jitter(Abs)', 'Jitter:RAP', 'Jitter:PPQ5', 'Jitter:DDP',
    'Shimmer', 'Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5',
    'Shimmer:APQ11', 'Shimmer:DDA',
    'NHR', 'HNR', 'RPDE', 'DFA', 'PPE'
]
TARGETS = ['motor_UPDRS', 'total_UPDRS']
GROUPS  = df['subject#'].values   # patient IDs used for group-aware split

X = df[FEATURE_COLS].values

# ── Patient-level split (GroupShuffleSplit) ───────────────────────────────
# test_size=0.2 → ~8–9 of the 42 patients go to test set
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, groups=GROUPS))

X_train, X_test = X[train_idx], X[test_idx]
train_patients  = df['subject#'].iloc[train_idx].nunique()
test_patients   = df['subject#'].iloc[test_idx].nunique()
print(f"Train: {len(train_idx)} recordings from {train_patients} patients")
print(f"Test : {len(test_idx)} recordings from {test_patients} patients")
print(f"No patient overlap: {set(df['subject#'].iloc[train_idx]) & set(df['subject#'].iloc[test_idx])}")

# ── Helper ────────────────────────────────────────────────────────────────
def evaluate(y_true, y_pred, model_name, target_name):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    return {'Model': model_name, 'Target': target_name,
            'MAE': round(mae, 3), 'RMSE': round(rmse, 3), 'R²': round(r2, 3)}

# ── Feature scaling (for linear models) ───────────────────────────────────
scaler       = StandardScaler()
X_train_sc   = scaler.fit_transform(X_train)
X_test_sc    = scaler.transform(X_test)

results = []

for target in TARGETS:
    y_train = df[target].values[train_idx]
    y_test  = df[target].values[test_idx]

    # Model 1 – Linear Regression (baseline)
    lr = LinearRegression()
    lr.fit(X_train_sc, y_train)
    results.append(evaluate(y_test, lr.predict(X_test_sc), 'Linear Regression', target))

    # Model 2 – Ridge Regression
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train_sc, y_train)
    results.append(evaluate(y_test, ridge.predict(X_test_sc), 'Ridge Regression', target))

    # Model 3 – Lasso Regression
    lasso = Lasso(alpha=0.1, max_iter=10000)
    lasso.fit(X_train_sc, y_train)
    n_zero = np.sum(lasso.coef_ == 0)
    print(f"  Lasso ({target}) zeroed {n_zero}/{len(lasso.coef_)} features")
    results.append(evaluate(y_test, lasso.predict(X_test_sc), 'Lasso Regression', target))

    # Model 4 – Random Forest (scale-invariant; use raw features)
    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    results.append(evaluate(y_test, rf.predict(X_test), 'Random Forest', target))

# ── Results table ─────────────────────────────────────────────────────────
results_df = pd.DataFrame(results)
print()
print(results_df.to_string(index=False))

# ── Save results CSV ──────────────────────────────────────────────────────
RESULTS_CSV = 'updrs_model_results.csv'
results_df.to_csv(RESULTS_CSV, index=False)

DRIVE_PATH = 'C:\\Users\\ssule\\OneDrive\\Desktop\\ProjectResultsBIFX546'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(DRIVE_PATH, exist_ok=True)
    results_df.to_csv(os.path.join(DRIVE_PATH, RESULTS_CSV), index=False)
    print(f"Results saved to path: {DRIVE_PATH}/{RESULTS_CSV}")
else:
    try:
        from google.colab import files
        files.download(RESULTS_CSV)
        print("Results CSV downloaded directly (Drive not mounted).")
    except ImportError:
        # Local environment - save to local path
        os.makedirs(DRIVE_PATH, exist_ok=True)
        results_df.to_csv(os.path.join(DRIVE_PATH, RESULTS_CSV), index=False)
        print(f"Results saved locally to: {os.path.join(DRIVE_PATH, RESULTS_CSV)}")
